In [0]:
url = (
    "https://api.worldbank.org/v2/"
    "country/BRA/indicator/NY.GDP.MKTP.CD"
    "?format=json&date=2000:2024&per_page=100"
)

print(url)

In [0]:
import requests

response = requests.get(url, timeout=30)

print(response.status_code)

In [0]:
payload = response.json()

print(type(payload))
print(len(payload))

In [0]:
metadata = payload[0]

print(metadata)

In [0]:
observations = payload[1]

print(observations[0])

In [0]:
print(len(observations))

In [0]:
print(observations[-1])

In [0]:
valores_ausentes = sum(
    1 for observation in observations
    if observation["value"] is None
)

print(valores_ausentes)

In [0]:
dados_pib = [
    {
        "codigo_pais": observation["countryiso3code"],
        "pais": observation["country"]["value"],
        "ano": int(observation["date"]),
        "pib_usd": observation["value"]
    }
    for observation in observations
]

print(dados_pib[0])

In [0]:
df_pib_brasil = spark.createDataFrame(dados_pib)

display(df_pib_brasil)

In [0]:
df_pib_brasil.printSchema()

In [0]:
from pyspark.sql import functions as F

df_pib_brasil_ordenado = df_pib_brasil.orderBy(F.col("ano").asc())

display(df_pib_brasil_ordenado)

In [0]:
df_pib_brasil_formatado = (
    df_pib_brasil_ordenado
    .withColumn(
        "pib_bilhoes_usd",
        F.round(F.col("pib_usd") / F.lit(1_000_000_000), 2)
    )
)

display(
    df_pib_brasil_formatado.select(
        "ano",
        "codigo_pais",
        "pais",
        "pib_bilhoes_usd"
    )
)

In [0]:
from pyspark.sql.window import Window

janela_ano = Window.orderBy("ano")

df_pib_com_variacao = (
    df_pib_brasil_formatado
    .withColumn(
        "pib_ano_anterior_usd",
        F.lag("pib_usd").over(janela_ano)
    )
    .withColumn(
        "variacao_anual_pct",
        F.round(
            (
                (F.col("pib_usd") - F.col("pib_ano_anterior_usd"))
                / F.col("pib_ano_anterior_usd")
            ) * 100,
            2
        )
    )
)

display(
    df_pib_com_variacao.select(
        "ano",
        "pib_bilhoes_usd",
        "variacao_anual_pct"
    )
)

In [0]:
df_variacoes_validas = (
    df_pib_com_variacao
    .filter(F.col("variacao_anual_pct").isNotNull())
)

maior_variacao = (
    df_variacoes_validas
    .orderBy(F.col("variacao_anual_pct").desc())
    .select("ano", "variacao_anual_pct")
    .first()
)

menor_variacao = (
    df_variacoes_validas
    .orderBy(F.col("variacao_anual_pct").asc())
    .select("ano", "variacao_anual_pct")
    .first()
)

print("Maior variação:", maior_variacao)
print("Menor variação:", menor_variacao)

In [0]:
janela_pais_ano = (
    Window
    .partitionBy("codigo_pais")
    .orderBy("ano")
)

print(janela_pais_ano)

In [0]:
df_pib_com_variacao = (
    df_pib_brasil_formatado
    .withColumn(
        "pib_ano_anterior_usd",
        F.lag("pib_usd").over(janela_pais_ano)
    )
    .withColumn(
        "variacao_anual_pct",
        F.round(
            (
                (F.col("pib_usd") - F.col("pib_ano_anterior_usd"))
                / F.col("pib_ano_anterior_usd")
            ) * 100,
            2
        )
    )
)

display(
    df_pib_com_variacao.select(
        "codigo_pais",
        "ano",
        "pib_bilhoes_usd",
        "variacao_anual_pct"
    )
)

In [0]:
quantidade_registros = df_pib_com_variacao.count()

print(quantidade_registros)

In [0]:
quantidade_colunas = len(df_pib_com_variacao.columns)

print(quantidade_colunas)

In [0]:
print(df_pib_com_variacao.columns)

In [0]:
df_pib_com_variacao.select(
    [
        F.sum(F.col(coluna).isNull().cast("int")).alias(coluna)
        for coluna in df_pib_com_variacao.columns
    ]
).display()

In [0]:
df_pib_com_variacao.filter(
    F.col("pib_ano_anterior_usd").isNull()
    | F.col("variacao_anual_pct").isNull()
).display()

In [0]:
assert quantidade_registros == 25, "Quantidade inesperada de registros"

assert (
    df_pib_com_variacao
    .filter(F.col("pib_usd").isNull())
    .count()
    == 0
), "Foram encontrados valores nulos no PIB"

print("Validações concluídas com sucesso!")